## A naive timing attempt & Coarse timing with ```perf_counter```

In [ ]:
import numpy as np

def make_problem(n):
    x = np.linspace(-10.0, 10.0, n)
    dx = x[1] - x[0]
    u = np.tanh(x)
    return x, u, dx

def residual(u, dx):
    r = np.empty_like(u)
    r[0] = 0.0
    r[-1] = 0.0
    r[1:-1] = (
        -(u[:-2] - 2*u[1:-1] + u[2:]) / dx**2
        + u[1:-1]*(u[1:-1]**2 - 1.0)
    )
    return r

In [ ]:
import time

start = time.time()

x, u, dx = make_problem(100_000)
r = residual(u, dx)

end = time.time()
print(end - start)

In [ ]:
from time import perf_counter

start = perf_counter()
x, u, dx = make_problem(100_000)
r = residual(u, dx)
elapsed = perf_counter() - start

print(f"elapsed time: {elapsed:.6f} s")

In [ ]:
import time


def measure_clock_resolution(clock, samples=1_000_000):
    previous = clock()
    smallest_difference = float("inf")
    changes_detected = 0

    for _ in range(samples):
        current = clock()
        difference = current - previous

        if difference > 0:
            changes_detected += 1
            smallest_difference = min(smallest_difference, difference)

        previous = current

    return smallest_difference, changes_detected


time_resolution, time_changes = measure_clock_resolution(time.time)
perf_resolution, perf_changes = measure_clock_resolution(time.perf_counter)


print("\ntime.time()")
print(f"Smallest positive difference: {time_resolution:.12f} seconds")
print(f"Detected changes:             {time_changes:,}")

print("\ntime.perf_counter()")
print(f"Smallest positive difference: {perf_resolution:.12f} seconds")
print(f"Detected changes:             {perf_changes:,}")

print("\n=== Resolution reported by Python ===")

time_info = time.get_clock_info("time")
perf_info = time.get_clock_info("perf_counter")

print(f"time.time():         {time_info.resolution:.12f} seconds")
print(f"time.perf_counter(): {perf_info.resolution:.12f} seconds")

```time.time()``` returns the number of seconds since 1970, so at present this is a value of the order of 1_780_000_000

In [ ]:
import math
import time

current_time = time.time()

print(f"Current time: {current_time}")
print(f"Float spacing: {math.ulp(current_time):.12f} seconds")
print(f"Float spacing: {math.ulp(current_time) * 1e9:.3f} nanoseconds")

In [ ]:
import math
import time

wall_clock = time.time()
performance_clock = time.perf_counter()

print("time.time()")
print(f"Value:       {wall_clock}")
print(f"Float ULP:   {math.ulp(wall_clock):.12f} seconds")
print(f"Float ULP:   {math.ulp(wall_clock) * 1e9:.3f} ns")

print("\ntime.perf_counter()")
print(f"Value:       {performance_clock}")
print(f"Float ULP:   {math.ulp(performance_clock):.12f} seconds")
print(f"Float ULP:   {math.ulp(performance_clock) * 1e9:.3f} ns")

## Repeating measurements

In [ ]:
from time import perf_counter
import numpy as np

def measure_once(func):
    start = perf_counter()
    result = func()
    elapsed = perf_counter() - start
    return elapsed, result

times = []
for _ in range(10):
    elapsed, r = measure_once(lambda: residual(u, dx))
    times.append(elapsed)

times = np.array(times)

print(times.min(), np.median(times), times.max())

## ```timeit``` for small statements

In [ ]:
import timeit
import numpy as np

x, u, dx = make_problem(100_000)

samples = timeit.repeat(
    "residual(u, dx)",
    globals={"residual": residual, "u": u, "dx": dx},
    number=100,
    repeat=7,
)

per_call = np.array(samples) / 100

print("best:", per_call.min())
print("median:", np.median(per_call))

## Timing across sizes: the first decision point

In [ ]:
sizes = [1_000, 3_000, 10_000, 30_000, 100_000]
best_times = []

for n in sizes:
    x, u, dx = make_problem(n)
    samples = timeit.repeat(
        "residual(u, dx)",
        globals={"residual": residual, "u": u, "dx": dx},
        number=50,
        repeat=5,)
    best_times.append(min(samples) / 50)

for n, t in zip(sizes, best_times):
    print(f"{n:8d}  {t:.6e} s")

## Profiling from Python

In [ ]:
import numpy as np

def make_problem(n):
    x = np.linspace(-10.0, 10.0, n)
    dx = x[1] - x[0]
    u = np.tanh(x)
    return x, u, dx

def residual(u, dx):
    r = np.empty_like(u)
    r[0] = 0.0
    r[-1] = 0.0
    r[1:-1] = (
        -(u[:-2] - 2*u[1:-1] + u[2:]) / dx**2
        + u[1:-1]*(u[1:-1]**2 - 1.0)
    )
    return r
    
def step(u, dx, dt):
    r = residual(u, dx)
    return u - dt*r

def run_simulation(n=1000, n_steps=200):
    x, u, dx = make_problem(n)

    for _ in range(n_steps):
        u = step(u, dx, dt=1e-10)

    return u

In [ ]:
import cProfile
import pstats
import io
from pstats import SortKey

profiler = cProfile.Profile()
profiler.enable()
u_final = run_simulation(n=100, n_steps=1000)
profiler.disable()
stream = io.StringIO()
stats = pstats.Stats(profiler, stream=stream)
stats.strip_dirs()
stats.sort_stats(SortKey.CUMULATIVE)
stats.print_stats()
print(stream.getvalue())

In [ ]:
import cProfile
import pstats
from pstats import SortKey

profiler = cProfile.Profile()

profiler.enable()
u_final = run_simulation(n=1000, n_steps=10000)
profiler.disable()

stats = pstats.Stats(profiler)
stats.strip_dirs()

stats.sort_stats(SortKey.CUMULATIVE)
stats.print_stats(15)

stats.sort_stats(SortKey.TIME)
stats.print_stats(15)

In [ ]:
def residual(u, dx):
    r = np.empty_like(u)
    r[0] = 0.0
    r[-1] = 0.0
    r[1:-1] = (-(u[:-2] - 2*u[1:-1] + u[2:]) / dx**2 + u[1:-1]*(u[1:-1]**2 - 1.0))
    return r

%load_ext line_profiler
%lprun -f residual run_simulation(n=10000, n_steps=10000)

## Estimate memory before running

In [ ]:
import numpy as np

def array_size_MB(shape, dtype=np.float64):
    n_values = np.prod(shape)
    itemsize = np.dtype(dtype).itemsize
    return n_values * itemsize / 1024**2

print(array_size_MB((2000, 2000)))       
print(array_size_MB((100, 2000, 2000)))

## Inspect actual array sizes

In [ ]:
x = np.linspace(-10.0, 10.0, 2_000_000)
u = np.tanh(x)

print(u.shape)
print(u.dtype)
print(u.nbytes / 1024**2, "MB")

## Tracing current and peak memory

In [ ]:
import tracemalloc

tracemalloc.start()

u_final = run_simulation(n=100000, n_steps=10000)

current, peak = tracemalloc.get_traced_memory()
tracemalloc.stop()

print(f"current: {current / 1024**2:.2f} MB")
print(f"peak:    {peak / 1024**2:.2f} MB")